<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 14


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Supplier в C#, который будет представлять информацию о
поставщиках товаров или услуг. На основе этого класса разработать 2-3
производных класса, демонстрирующих принципы наследования и полиморфизма.
В каждом из классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text.Json;

namespace SupplierManagement
{
    public interface ISupplierNotifier
    {
        void SendAlert(string message);
        bool CanSend { get; }
    }

    public interface IDataExporter
    {
        string ToJson();
        string ToCsv();
    }

    public interface IPaymentHandler
    {
        bool HandlePayment(decimal amount, string currency);
    }

    public abstract class SupplierBase : ISupplierNotifier, IDataExporter
    {
        public int Id { get; }
        public string Name { get; }
        public string ContactPhone { get; set; }
        public string ContactEmail { get; set; }
        public decimal SuccessRate { get; private set; }
        public bool IsOperational { get; set; } = true;
        public DateTime RegistrationDate { get; }
        public List<string> Contacts { get; } = new();

        protected SupplierBase(int id, string name, string phone, string email)
        {
            Id = id;
            Name = name ?? throw new ArgumentNullException(nameof(name));
            ContactPhone = phone;
            ContactEmail = email;
            RegistrationDate = DateTime.Now;
        }

        public virtual void ShowInfo()
        {
            Console.WriteLine($"#{Id} {Name} (рейтинг: {SuccessRate:F1})");
        }

        public abstract decimal ComputeDiscount(decimal orderAmount);

        public void HandleOrder(string orderDetails, decimal amount)
        {
            var discount = ComputeDiscount(amount);
            Console.WriteLine($"{Name} -> заказ: {orderDetails}");
            Console.WriteLine($"Скидка: {discount:P1}");
        }

        public void AdjustSuccessRate(decimal newRating)
        {
            SuccessRate = (SuccessRate + newRating) / 2;
            Console.WriteLine($"Обновлен рейтинг {Name}: {SuccessRate:F1}");
        }

        public virtual bool IsValid()
        {
            return !string.IsNullOrWhiteSpace(Name) &&
                   !string.IsNullOrWhiteSpace(ContactEmail) &&
                   !string.IsNullOrWhiteSpace(ContactPhone);
        }

        public void RegisterContact(string contactName)
        {
            Contacts.Add(contactName);
            Console.WriteLine($"Добавлен контакт: {contactName}");
        }

        bool ISupplierNotifier.CanSend => !string.IsNullOrWhiteSpace(ContactEmail) && IsOperational;

        void ISupplierNotifier.SendAlert(string message)
        {
            if (((ISupplierNotifier)this).CanSend)
            {
                Console.WriteLine($"🔔 {Name}: {message}");
            }
        }

        string IDataExporter.ToJson()
        {
            var data = new { Id, Name, ContactEmail, SuccessRate, IsOperational };
            return JsonSerializer.Serialize(data, new JsonSerializerOptions { WriteIndented = true });
        }

        string IDataExporter.ToCsv()
        {
            return $"{Id},\"{Name}\",{ContactEmail},{SuccessRate:F2},{IsOperational}";
        }
    }

    public class GoodsSupplier : SupplierBase, IPaymentHandler
    {
        public string ProductCategory { get; }
        public int AvailableStock { get; private set; }
        public HashSet<string> ProductLines { get; } = new();
        public string StorageLocation { get; set; }
        public bool AcceptsLargeOrders { get; set; }
        public decimal DeliveryFee { get; set; } = 300;

        public GoodsSupplier(int id, string name, string phone, string email, string category)
            : base(id, name, phone, email)
        {
            ProductCategory = category ?? "Разное";
        }

        public override void ShowInfo()
        {
            base.ShowInfo();
            Console.WriteLine($"   Категория: {ProductCategory}, Позиций: {ProductLines.Count}");
            if (!string.IsNullOrEmpty(StorageLocation))
                Console.WriteLine($"   Склад: {StorageLocation}");
        }

        public override decimal ComputeDiscount(decimal orderAmount)
        {
            var discount = 0m;
            
            if (orderAmount > 15000) discount += 0.04m;
            if (orderAmount > 45000) discount += 0.07m;
            if (AcceptsLargeOrders && orderAmount > 90000) discount += 0.04m;
            
            return discount;
        }

        public void RegisterProduct(string productName)
        {
            if (ProductLines.Add(productName))
            {
                AvailableStock += 15;
                Console.WriteLine($"✅ {Name} + товар: {productName}");
            }
        }

        public void ModifyStock(int quantityChange)
        {
            AvailableStock += quantityChange;
            Console.WriteLine($"📦 {Name}: запас изменен на {quantityChange}");
        }

        public bool CheckStock(string product, int requiredQty)
        {
            return ProductLines.Contains(product) && AvailableStock >= requiredQty;
        }

        public bool HandlePayment(decimal amount, string currency)
        {
            var processingFee = amount * 0.018m;
            Console.WriteLine($"💰 {Name}: платеж {amount + processingFee:C} ({currency})");
            return true;
        }
    }

    public class ServiceProvider : SupplierBase, IPaymentHandler
    {
        public string ServiceCategory { get; }
        public decimal RatePerHour { get; set; }
        public List<string> ServiceAreas { get; } = new();
        public int SpecialistsCount { get; set; } = 2;
        public bool ProvidesUrgentSupport { get; set; }
        public int MaxResponseHours { get; set; } = 24;

        public ServiceProvider(int id, string name, string phone, string email, string serviceType)
            : base(id, name, phone, email)
        {
            ServiceCategory = serviceType ?? "Общие услуги";
        }

        public override void ShowInfo()
        {
            base.ShowInfo();
            Console.WriteLine($"   Услуги: {ServiceCategory}, Ставка: {RatePerHour:C}/час");
            Console.WriteLine($"   Специалистов: {SpecialistsCount}");
        }

        public override decimal ComputeDiscount(decimal orderAmount)
        {
            return orderAmount > 18000 ? 0.12m : 0.08m;
        }

        public void AddServiceArea(string area)
        {
            ServiceAreas.Add(area);
            Console.WriteLine($"🌍 {Name} + регион: {area}");
        }

        public decimal QuoteProject(int estimatedHours)
        {
            var total = estimatedHours * RatePerHour;
            return estimatedHours > 50 ? total * 0.88m : total;
        }

        public bool CanProvideUrgentService()
        {
            return ProvidesUrgentSupport && MaxResponseHours <= 6;
        }

        public bool HandlePayment(decimal amount, string currency)
        {
            Console.WriteLine($"💳 {Name}: оплата услуг {amount:C} ({currency})");
            return true;
        }
    }

    public class EcoFriendlySupplier : GoodsSupplier
    {
        public bool HasEcoCertificate { get; set; }
        public decimal EnvironmentalImpact { get; private set; }
        public bool UsesGreenEnergy { get; set; }
        public decimal WasteRecyclingRatio { get; set; } = 0.75m;

        public EcoFriendlySupplier(int id, string name, string phone, string email, string category)
            : base(id, name, phone, email, category) { }

        public override void ShowInfo()
        {
            base.ShowInfo();
            var certStatus = HasEcoCertificate ? "сертифицирован" : "без сертификата";
            Console.WriteLine($"   Экология: {certStatus}, Влияние: {EnvironmentalImpact:F2}кг CO₂");
        }

        public override decimal ComputeDiscount(decimal orderAmount)
        {
            var baseDiscount = base.ComputeDiscount(orderAmount);
            
            if (HasEcoCertificate) baseDiscount += 0.025m;
            if (UsesGreenEnergy) baseDiscount += 0.015m;
            if (WasteRecyclingRatio > 0.8m) baseDiscount += 0.01m;
            
            return baseDiscount;
        }

        public void ImproveEcoScore()
        {
            EnvironmentalImpact *= 0.92m;
            Console.WriteLine($"🌱 {Name}: улучшены экологические показатели");
        }

        public decimal CalculateSustainabilityBonus()
        {
            var bonus = (1 - EnvironmentalImpact) * 120;
            return UsesGreenEnergy ? bonus * 1.5m : bonus;
        }
    }

    public class SupplierCoordinator
    {
        private readonly List<SupplierBase> _suppliers;
        private readonly INotificationService _notifier;

        public SupplierCoordinator(INotificationService notifier)
        {
            _suppliers = new List<SupplierBase>();
            _notifier = notifier;
        }

        public void RegisterSupplier(SupplierBase supplier)
        {
            if (supplier.IsValid())
            {
                _suppliers.Add(supplier);
                _notifier.SendRegistrationAlert(supplier.Name, supplier.ContactEmail);
            }
        }

        public SupplierBase FindById(int id) => _suppliers.FirstOrDefault(s => s.Id == id);

        public IEnumerable<SupplierBase> GetActiveSuppliers() => _suppliers.Where(s => s.IsOperational);

        public void ExecutePayments(decimal amount, string currency)
        {
            var paymentHandlers = _suppliers.OfType<IPaymentHandler>().ToList();
            paymentHandlers.ForEach(handler => handler.HandlePayment(amount, currency));
        }
    }

    public interface INotificationService
    {
        void SendRegistrationAlert(string supplierName, string email);
    }

    public class SupplierNotificationService : INotificationService
    {
        public void SendRegistrationAlert(string supplierName, string email)
        {
            Console.WriteLine($"✉️  Зарегистрирован {supplierName} -> {email}");
        }
    }

    class Program
    {
        static void Main()
        {
            var notifier = new SupplierNotificationService();
            var coordinator = new SupplierCoordinator(notifier);

            var supplierList = new SupplierBase[]
            {
                new SupplierBase(101, "Поставщик-ОСНОВА", "+79990001122", "main@example.com"),
                new GoodsSupplier(102, "ТехноПоставки", "+79993334455", "tech@example.com", "Электроника"),
                new ServiceProvider(103, "ПрофУслуги", "+79996667788", "services@example.com", "IT-аутсорсинг"),
                new EcoFriendlySupplier(104, "ЭкоПродукт", "+79991234567", "eco@example.com", "Органические товары")
            };

            Array.ForEach(supplierList, coordinator.RegisterSupplier);

            // Настройка поставщиков
            var techSupplier = (GoodsSupplier)supplierList[1];
            techSupplier.RegisterProduct("Ноутбуки бизнес-класса");
            techSupplier.RegisterProduct("Мониторы 4K");
            techSupplier.StorageLocation = "СКЛАД-42";
            techSupplier.AcceptsLargeOrders = true;

            var serviceProvider = (ServiceProvider)supplierList[2];
            serviceProvider.RatePerHour = 1250;
            serviceProvider.AddServiceArea("Центральный регион");
            serviceProvider.SpecialistsCount = 6;
            serviceProvider.ProvidesUrgentSupport = true;

            var ecoSupplier = (EcoFriendlySupplier)supplierList[3];
            ecoSupplier.HasEcoCertificate = true;
            ecoSupplier.EnvironmentalImpact = 0.8m;
            ecoSupplier.UsesGreenEnergy = true;
            ecoSupplier.ImproveEcoScore();

            Console.WriteLine("\n" + new string('=', 50));
            Console.WriteLine("СПИСОК ПОСТАВЩИКОВ");
            Console.WriteLine(new string('=', 50));
            
            Array.ForEach(supplierList, s => { s.ShowInfo(); Console.WriteLine(); });

            Console.WriteLine("ОБРАБОТКА ЗАКАЗОВ");
            Console.WriteLine(new string('-', 30));
            
            var testOrders = new decimal[] { 8000, 22000, 68000, 35000 };
            for (int i = 0; i < supplierList.Length; i++)
            {
                supplierList[i].HandleOrder($"Тест-заказ {i + 1}", testOrders[i]);
            }

            Console.WriteLine("\nФИНАНСОВЫЕ ОПЕРАЦИИ");
            Console.WriteLine(new string('-', 30));
            coordinator.ExecutePayments(15000, "RUB");

            Console.WriteLine("\nДОПОЛНИТЕЛЬНЫЕ ОПЕРАЦИИ");
            Console.WriteLine(new string('-', 30));
            
            techSupplier.ModifyStock(25);
            Console.WriteLine($"Проверка наличия: {techSupplier.CheckStock("Ноутбуки бизнес-класса", 10)}");
            
            var projectQuote = serviceProvider.QuoteProject(60);
            Console.WriteLine($"Смета проекта: {projectQuote:C}");
            
            var ecoBonus = ecoSupplier.CalculateSustainabilityBonus();
            Console.WriteLine($"Эко-премия: {ecoBonus:C}");

            Console.WriteLine("\nУВЕДОМЛЕНИЯ");
            Console.WriteLine(new string('-', 30));
            foreach (var supplier in supplierList)
            {
                ((ISupplierNotifier)supplier).SendAlert("Проверка связи системы");
            }
        }
    }
}

// Крастотище ответ

In [ ]:
✉️  Зарегистрирован Поставщик-ОСНОВА -> main@example.com
✉️  Зарегистрирован ТехноПоставки -> tech@example.com
✉️  Зарегистрирован ПрофУслуги -> services@example.com
✉️  Зарегистрирован ЭкоПродукт -> eco@example.com
✅ ТехноПоставки + товар: Ноутбуки бизнес-класса
✅ ТехноПоставки + товар: Мониторы 4K
🌍 ПрофУслуги + регион: Центральный регион
🌱 ЭкоПродукт: улучшены экологические показатели

==================================================
СПИСОК ПОСТАВЩИКОВ
==================================================
#101 Поставщик-ОСНОВА (рейтинг: 0,0)

#102 ТехноПоставки (рейтинг: 0,0)
   Категория: Электроника, Позиций: 2
   Склад: СКЛАД-42

#103 ПрофУслуги (рейтинг: 0,0)
   Услуги: IT-аутсорсинг, Ставка: ₽1,250.00/час
   Специалистов: 6

#104 ЭкоПродукт (рейтинг: 0,0)
   Категория: Органические товары, Позиций: 0
   Экология: сертифицирован, Влияние: 0,74кг CO₂

ОБРАБОТКА ЗАКАЗОВ
------------------------------
Поставщик-ОСНОВА -> заказ: Тест-заказ 1
Скидка: 0,0%

ТехноПоставки -> заказ: Тест-заказ 2
Скидка: 4,0%

ТехноПоставки -> заказ: Тест-заказ 3
Скидка: 11,0%

ПрофУслуги -> заказ: Тест-заказ 4
Скидка: 12,0%


ФИНАНСОВЫЕ ОПЕРАЦИИ
------------------------------
💰 ТехноПоставки: платеж ₽15,270.00 (RUB)
💳 ПрофУслуги: оплата услуг ₽15,000.00 (RUB)
💳 ЭкоПродукт: платеж ₽15,270.00 (RUB)

ДОПОЛНИТЕЛЬНЫЕ ОПЕРАЦИИ
------------------------------
📦 ТехноПоставки: запас изменен на 25
Проверка наличия: True
Смета проекта: ₽66,000.00
Эко-премия: ₽31,20

УВЕДОМЛЕНИЯ
------------------------------
🔔 Поставщик-ОСНОВА: Проверка связи системы
🔔 ТехноПоставки: Проверка связи системы
🔔 ПрофУслуги: Проверка связи системы
🔔 ЭкоПродукт: Проверка связи системы

